In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [7]:
!pip install cyvcf2
from cyvcf2 import VCF

In [8]:
vcf_path = "/content/drive/MyDrive/FYP_DATA/raw_data/1000genomes_annotated/1kg_chr1_annotated.vcf.gz"
vcf = VCF(vcf_path)


In [9]:
print("==================== INFO FIELDS ====================")

for line in vcf.raw_header.split("\n"):
    if line.startswith("##INFO="):
        print(line)

==================== INFO FIELDS ====================
##INFO=<ID=AF,Number=A,Type=Float,Description="Estimated allele frequency in the range (0,1)">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Total number of alternate alleles in called genotypes">
##INFO=<ID=NS,Number=1,Type=Integer,Description="Number of samples with data">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Total number of alleles in called genotypes">
##INFO=<ID=EAS_AF,Number=A,Type=Float,Description="Allele frequency in the EAS populations calculated from AC and AN, in the range (0,1)">
##INFO=<ID=EUR_AF,Number=A,Type=Float,Description="Allele frequency in the EUR populations calculated from AC and AN, in the range (0,1)">
##INFO=<ID=AFR_AF,Number=A,Type=Float,Description="Allele frequency in the AFR populations calculated from AC and AN, in the range (0,1)">
##INFO=<ID=AMR_AF,Number=A,Type=Float,Description="Allele frequency in the AMR populations calculated from AC and AN, in the range (0,1)">
##INFO=<ID=SAS_A

In [10]:
vcf = VCF(vcf_path)

# Print the first few header lines
for line in vcf.raw_header.split('\n')[:20]:  # adjust number if you want more
    print(line)

##fileformat=VCFv4.3
##FILTER=<ID=PASS,Description="All filters passed">
##fileDate=26022019_15h52m43s
##source=IGSRpipeline
##reference=ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa
##FORMAT=<ID=GT,Number=1,Type=String,Description="Phased Genotype">
##contig=<ID=1>
##INFO=<ID=AF,Number=A,Type=Float,Description="Estimated allele frequency in the range (0,1)">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Total number of alternate alleles in called genotypes">
##INFO=<ID=NS,Number=1,Type=Integer,Description="Number of samples with data">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Total number of alleles in called genotypes">
##INFO=<ID=EAS_AF,Number=A,Type=Float,Description="Allele frequency in the EAS populations calculated from AC and AN, in the range (0,1)">
##INFO=<ID=EUR_AF,Number=A,Type=Float,Description="Allele frequency in the EUR populations calculated from AC and AN, in the range (0,1)">

In [11]:
# Print all INFO fields to find CSQ
for line in vcf.raw_header.split('\n'):
    if 'CSQ' in line:
        print(line)

##INFO=<ID=CSQ,Number=.,Type=String,Description="Consequence annotations from Ensembl VEP. Format: Allele|Consequence|IMPACT|SYMBOL|Gene|Feature_type|Feature|BIOTYPE|EXON|INTRON|HGVSc|HGVSp|cDNA_position|CDS_position|Protein_position|Amino_acids|Codons|Existing_variation|DISTANCE|STRAND|FLAGS|SYMBOL_SOURCE|HGNC_ID|CANONICAL">


In [13]:
import pandas as pd

In [14]:
vcf = VCF(vcf_path)

consequences_set = set()

for var in vcf:
    csqs = var.INFO.get('CSQ')
    if not csqs:
        continue
    for csq in csqs.split(','):
        fields = csq.split('|')
        # Consequence is the 2nd field
        for consequence in fields[1].split('&'):  # some variants have multiple consequences separated by '&'
            consequences_set.add(consequence)

# Show all unique consequences
print("Total unique consequences:", len(consequences_set))
print(sorted(consequences_set))

Total unique consequences: 29
['3_prime_UTR_variant', '5_prime_UTR_variant', 'NMD_transcript_variant', 'coding_sequence_variant', 'downstream_gene_variant', 'frameshift_variant', 'incomplete_terminal_codon_variant', 'inframe_deletion', 'inframe_insertion', 'intergenic_variant', 'intron_variant', 'mature_miRNA_variant', 'missense_variant', 'non_coding_transcript_exon_variant', 'non_coding_transcript_variant', 'protein_altering_variant', 'splice_acceptor_variant', 'splice_donor_5th_base_variant', 'splice_donor_region_variant', 'splice_donor_variant', 'splice_polypyrimidine_tract_variant', 'splice_region_variant', 'start_lost', 'start_retained_variant', 'stop_gained', 'stop_lost', 'stop_retained_variant', 'synonymous_variant', 'upstream_gene_variant']


In [15]:
# ['3_prime_UTR_variant', '5_prime_UTR_variant', 'NMD_transcript_variant', 'coding_sequence_variant', 'downstream_gene_variant', 'frameshift_variant', 'incomplete_terminal_codon_variant', 'inframe_deletion', 'inframe_insertion', 'intergenic_variant', 'intron_variant', 'mature_miRNA_variant', 'missense_variant', 'non_coding_transcript_exon_variant', 'non_coding_transcript_variant', 'protein_altering_variant', 'splice_acceptor_variant', 'splice_donor_5th_base_variant', 'splice_donor_region_variant', 'splice_donor_variant', 'splice_polypyrimidine_tract_variant', 'splice_region_variant', 'start_lost', 'start_retained_variant', 'stop_gained', 'stop_lost', 'stop_retained_variant', 'synonymous_variant', 'upstream_gene_variant']


In [16]:
from cyvcf2 import VCF, Writer

In [17]:
vcf_path = '/content/drive/MyDrive/FYP_DATA/raw_data/1000genomes_annotated/1kg_chr1_annotated.vcf.gz'
output_vcf = '/content/drive/MyDrive/FYP_DATA/processed_data/1k/annotated_processing/chr1_missense_only.vcf.gz'

vcf = VCF(vcf_path)
w = Writer(output_vcf, vcf)  # preserves header and sample info

for var in vcf:
    csqs = var.INFO.get('CSQ')
    if not csqs:
        continue
    # keep only variants where at least one transcript is missense
    if any('missense_variant' in csq.split('|')[1] for csq in csqs.split(',')):
        w.write_record(var)

w.close()
vcf.close()
print("Missense-only VCF created successfully!")


Missense-only VCF created successfully!


In [18]:
vcf = VCF(output_vcf)
# Print first 20 header lines
for line in vcf.raw_header.split('\n')[:20]:
    print(line)

vcf.close()

##fileformat=VCFv4.3
##FILTER=<ID=PASS,Description="All filters passed">
##fileDate=26022019_15h52m43s
##source=IGSRpipeline
##reference=ftp://ftp.1000genomes.ebi.ac.uk/vol1/ftp/technical/reference/GRCh38_reference_genome/GRCh38_full_analysis_set_plus_decoy_hla.fa
##FORMAT=<ID=GT,Number=1,Type=String,Description="Phased Genotype">
##contig=<ID=1>
##INFO=<ID=AF,Number=A,Type=Float,Description="Estimated allele frequency in the range (0,1)">
##INFO=<ID=AC,Number=A,Type=Integer,Description="Total number of alternate alleles in called genotypes">
##INFO=<ID=NS,Number=1,Type=Integer,Description="Number of samples with data">
##INFO=<ID=AN,Number=1,Type=Integer,Description="Total number of alleles in called genotypes">
##INFO=<ID=EAS_AF,Number=A,Type=Float,Description="Allele frequency in the EAS populations calculated from AC and AN, in the range (0,1)">
##INFO=<ID=EUR_AF,Number=A,Type=Float,Description="Allele frequency in the EUR populations calculated from AC and AN, in the range (0,1)">

In [20]:
vcf = VCF(output_vcf)

print("First 5 variants:")
for i, var in enumerate(vcf):
    print(f"{var.CHROM}\t{var.POS}\t{var.REF}\t{','.join(var.ALT)}\t{var.INFO.get('CSQ')}")
    if i == 4:
        break

vcf.close()


First 5 variants:
1	69511	A	G	G|missense_variant|MODERATE|OR4F5|ENSG00000186092|Transcript|ENST00000641515|protein_coding|3/3||||544|484|162|T/A|Aca/Gca|||1||HGNC|HGNC:14825|YES
1	69541	A	G	G|missense_variant|MODERATE|OR4F5|ENSG00000186092|Transcript|ENST00000641515|protein_coding|3/3||||574|514|172|S/G|Agc/Ggc|||1||HGNC|HGNC:14825|YES
1	69555	T	G	G|missense_variant|MODERATE|OR4F5|ENSG00000186092|Transcript|ENST00000641515|protein_coding|3/3||||588|528|176|F/L|ttT/ttG|||1||HGNC|HGNC:14825|YES
1	924499	C	A	A|upstream_gene_variant|MODIFIER|SAMD11|ENSG00000187634|Transcript|ENST00000342066|protein_coding|||||||||||1232|1||HGNC|HGNC:28706|,A|upstream_gene_variant|MODIFIER|LINC02593|ENSG00000223764|Transcript|ENST00000417705|lncRNA|||||||||||4807|-1||HGNC|HGNC:53933|,A|upstream_gene_variant|MODIFIER|SAMD11|ENSG00000187634|Transcript|ENST00000437963|protein_coding|||||||||||651|1|cds_end_NF|HGNC|HGNC:28706|,A|upstream_gene_variant|MODIFIER|LINC02593|ENSG00000223764|Transcript|ENST00000609207